# Análisis de Métricas de Performance

**Objetivo:** Medir y analizar métricas de rendimiento del sistema de análisis de complejidades
**Duración estimada:** 35 minutos

---

## Contenido

1. [Setup](#setup)
2. [Recolección de Métricas](#recoleccion-de-metricas)
3. [Tiempos de Procesamiento por Módulo](#tiempos-de-procesamiento-por-modulo)
4. [Análisis de Uso de Memoria](#analisis-de-uso-de-memoria)
5. [Throughput del Sistema](#throughput-del-sistema)
6. [Comparativas por Complejidad de Algoritmo](#comparativas-por-complejidad-de-algoritmo)
7. [Visualización de Métricas](#visualizacion-de-metricas)
8. [Resumen y Conclusiones](#resumen-y-conclusiones)

---

## 1. Setup

In [ ]:
import sys
import time
import json
import statistics
from pathlib import Path

sys.path.insert(0, '../..')

from app.core.parser import parse_pseudocode
from app.core.analyzer import AnalyzerEngine
from app.core.patterns import PatternDetector
from app.core.data_structures import StructureIdentifier
from app.profiling import (
    get_performance_monitor,
    enable_profiling,
    PerformanceLevel
)

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

enable_profiling(enable_timing=True, enable_memory=True)
monitor = get_performance_monitor(enable_memory=True)

print("Setup completado")

---

## 2. Recolección de Métricas

El módulo de profiling permite recolectar métricas detalladas de cada operación del sistema.
Las métricas se agrupan por módulo y operación para facilitar el análisis comparativo.

### Algoritmos de Referencia para Benchmarking

In [ ]:
BENCHMARK_ALGORITHMS = {
    "O(1)_constante": """
algorithm acceso(A[], i)
begin
    return A[i]
end
""",
    "O(n)_lineal": """
algorithm sumaTotal(A[], n)
begin
    total <- 0
    for i <- 1 to n do
        total <- total + A[i]
    end
    return total
end
""",
    "O(n2)_cuadratico": """
algorithm bubbleSort(A[], n)
begin
    for i <- 1 to n - 1 do
        for j <- 1 to n - i do
            if (A[j] > A[j + 1]) then
                temp <- A[j]
                A[j] <- A[j + 1]
                A[j + 1] <- temp
            end
        end
    end
end
""",
    "O(nlogn)_merge": """
algorithm mergeSort(A[], p, r)
begin
    if (p < r) then
        q <- floor((p + r) / 2)
        call mergeSort(A, p, q)
        call mergeSort(A, q + 1, r)
    end
end
""",
    "O(2n)_exponencial": """
algorithm fibonacci(n)
begin
    if (n <= 1) then
        return n
    end
    return fibonacci(n - 1) + fibonacci(n - 2)
end
"""
}

print(f"Algoritmos de benchmark: {list(BENCHMARK_ALGORITHMS.keys())}")

### Función de Medición

In [ ]:
def medir_pipeline_completo(nombre, codigo, repeticiones=5):
    """
    Mide el tiempo de cada etapa del pipeline para un algoritmo.
    
    Etapas medidas:
    - Parsing
    - Análisis de complejidad
    - Detección de patrones
    - Detección de estructuras
    """
    resultados = {
        "nombre": nombre,
        "parsing_ms": [],
        "analisis_ms": [],
        "patrones_ms": [],
        "estructuras_ms": [],
        "total_ms": []
    }
    
    engine = AnalyzerEngine()
    pattern_detector = PatternDetector()
    structure_identifier = StructureIdentifier()
    
    for _ in range(repeticiones):
        inicio_total = time.perf_counter()
        
        # Etapa 1: Parsing
        t0 = time.perf_counter()
        ast = parse_pseudocode(codigo)
        t1 = time.perf_counter()
        resultados["parsing_ms"].append((t1 - t0) * 1000)
        
        # Etapa 2: Análisis de complejidad
        t0 = time.perf_counter()
        analysis = engine.analyze(ast)
        t1 = time.perf_counter()
        resultados["analisis_ms"].append((t1 - t0) * 1000)
        
        # Etapa 3: Detección de patrones
        t0 = time.perf_counter()
        patterns = pattern_detector.detect(ast)
        t1 = time.perf_counter()
        resultados["patrones_ms"].append((t1 - t0) * 1000)
        
        # Etapa 4: Detección de estructuras
        t0 = time.perf_counter()
        structures = structure_identifier.identify(ast)
        t1 = time.perf_counter()
        resultados["estructuras_ms"].append((t1 - t0) * 1000)
        
        fin_total = time.perf_counter()
        resultados["total_ms"].append((fin_total - inicio_total) * 1000)
    
    # Calcular estadísticas
    for etapa in ["parsing_ms", "analisis_ms", "patrones_ms", "estructuras_ms", "total_ms"]:
        tiempos = resultados[etapa]
        resultados[f"{etapa}_media"] = statistics.mean(tiempos)
        resultados[f"{etapa}_desv"] = statistics.stdev(tiempos) if len(tiempos) > 1 else 0
        resultados[f"{etapa}_min"] = min(tiempos)
        resultados[f"{etapa}_max"] = max(tiempos)
    
    return resultados

# Ejecutar benchmark
print("Ejecutando benchmark...")
resultados_benchmark = {}
for nombre, codigo in BENCHMARK_ALGORITHMS.items():
    print(f"  Midiendo: {nombre}")
    resultados_benchmark[nombre] = medir_pipeline_completo(nombre, codigo)

print("Benchmark completado")

---

## 3. Tiempos de Procesamiento por Módulo

In [ ]:
# Mostrar tabla de resultados
print(f"{'Algoritmo':<25} {'Parsing':>10} {'Análisis':>10} {'Patrones':>10} {'Estructuras':>12} {'Total':>10}")

for nombre, datos in resultados_benchmark.items():
    print(f"{nombre:<25} "
          f"{datos['parsing_ms_media']:>9.2f}ms "
          f"{datos['analisis_ms_media']:>9.2f}ms "
          f"{datos['patrones_ms_media']:>9.2f}ms "
          f"{datos['estructuras_ms_media']:>11.2f}ms "
          f"{datos['total_ms_media']:>9.2f}ms")


### Análisis de Dispersión por Etapa

In [ ]:
# Analizar variabilidad de cada etapa
print("\nVARIABILIDAD POR ETAPA (desviación estándar):")

etapas = ["parsing_ms", "analisis_ms", "patrones_ms", "estructuras_ms"]
etapas_nombres = ["Parsing", "Análisis", "Patrones", "Estructuras"]

for etapa, nombre_etapa in zip(etapas, etapas_nombres):
    desviaciones = [datos[f"{etapa}_desv"] for datos in resultados_benchmark.values()]
    print(f"{nombre_etapa:<15}: "
          f"min_desv={min(desviaciones):.3f}ms, "
          f"max_desv={max(desviaciones):.3f}ms, "
          f"prom_desv={statistics.mean(desviaciones):.3f}ms")

---

## 4. Análisis de Uso de Memoria

In [ ]:
import tracemalloc

def medir_memoria_pipeline(nombre, codigo):
    """Mide el uso de memoria pico durante el pipeline completo."""
    tracemalloc.start()
    snapshot_inicial = tracemalloc.take_snapshot()
    
    engine = AnalyzerEngine()
    pattern_detector = PatternDetector()
    structure_identifier = StructureIdentifier()
    
    # Pipeline completo
    ast = parse_pseudocode(codigo)
    analysis = engine.analyze(ast)
    patterns = pattern_detector.detect(ast)
    structures = structure_identifier.identify(ast)
    
    snapshot_final = tracemalloc.take_snapshot()
    tracemalloc.stop()
    
    # Comparar snapshots
    stats = snapshot_final.compare_to(snapshot_inicial, 'lineno')
    
    memoria_total_kb = sum(stat.size_diff for stat in stats if stat.size_diff > 0) / 1024
    
    return {
        "nombre": nombre,
        "memoria_total_kb": memoria_total_kb,
        "top_allocations": [
            {
                "archivo": str(stat.traceback[0]),
                "tamanio_kb": stat.size_diff / 1024
            }
            for stat in sorted(stats, key=lambda s: s.size_diff, reverse=True)[:5]
            if stat.size_diff > 0
        ]
    }

print("Midiendo uso de memoria...")
memoria_resultados = {}
for nombre, codigo in BENCHMARK_ALGORITHMS.items():
    memoria_resultados[nombre] = medir_memoria_pipeline(nombre, codigo)
    print(f"  {nombre}: {memoria_resultados[nombre]['memoria_total_kb']:.2f} KB")

---

## 5. Throughput del Sistema

In [ ]:
def calcular_throughput(algoritmos_por_segundo_objetivo=10):
    """
    Calcula cuántos algoritmos puede procesar el sistema por segundo.
    Simula carga con el algoritmo O(n^2) como caso representativo.
    """
    codigo_representativo = BENCHMARK_ALGORITHMS["O(n2)_cuadratico"]
    engine = AnalyzerEngine()
    pattern_detector = PatternDetector()
    structure_identifier = StructureIdentifier()
    
    tiempos = []
    n_iteraciones = 20
    
    for _ in range(n_iteraciones):
        inicio = time.perf_counter()
        ast = parse_pseudocode(codigo_representativo)
        analysis = engine.analyze(ast)
        patterns = pattern_detector.detect(ast)
        structures = structure_identifier.identify(ast)
        fin = time.perf_counter()
        tiempos.append(fin - inicio)
    
    tiempo_promedio_s = statistics.mean(tiempos)
    throughput = 1 / tiempo_promedio_s
    
    print(f"Tiempo promedio por algoritmo: {tiempo_promedio_s * 1000:.2f}ms")
    print(f"Throughput estimado: {throughput:.1f} algoritmos/segundo")
    print(f"Objetivo: {algoritmos_por_segundo_objetivo} algoritmos/segundo")
    
    if throughput >= algoritmos_por_segundo_objetivo:
        print(f"PASS: El sistema cumple el objetivo de throughput")
    else:
        print(f"WARN: El sistema está por debajo del objetivo")
    
    return throughput

throughput_actual = calcular_throughput()

---

## 6. Comparativas por Complejidad de Algoritmo

In [ ]:
# Analizar si el tiempo de análisis correlaciona con la complejidad del algoritmo
complejidades_orden = ["O(1)_constante", "O(n)_lineal", "O(nlogn)_merge", "O(n2)_cuadratico", "O(2n)_exponencial"]
complejidades_etiquetas = ["O(1)", "O(n)", "O(n log n)", "O(n^2)", "O(2^n)"]

tiempos_totales = [
    resultados_benchmark[c]["total_ms_media"]
    for c in complejidades_orden
    if c in resultados_benchmark
]

print("CORRELACIÓN COMPLEJIDAD vs TIEMPO DE ANÁLISIS:")
for etiqueta, tiempo in zip(complejidades_etiquetas, tiempos_totales):
    barra = "#" * int(tiempo / max(tiempos_totales) * 40)
    print(f"{etiqueta:<12}: {barra:<40} {tiempo:.2f}ms")

---

## 7. Visualización de Métricas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Métricas de Performance - Analizador de Complejidades", fontsize=14)

# Gráfico 1: Tiempo por etapa (stacked bar)
ax1 = axes[0, 0]
nombres_cortos = [n.split("_")[0] for n in complejidades_orden if n in resultados_benchmark]
etapas_datos = {
    "Parsing": [resultados_benchmark[c]["parsing_ms_media"] for c in complejidades_orden if c in resultados_benchmark],
    "Análisis": [resultados_benchmark[c]["analisis_ms_media"] for c in complejidades_orden if c in resultados_benchmark],
    "Patrones": [resultados_benchmark[c]["patrones_ms_media"] for c in complejidades_orden if c in resultados_benchmark],
    "Estructuras": [resultados_benchmark[c]["estructuras_ms_media"] for c in complejidades_orden if c in resultados_benchmark],
}
colores = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
bottom = np.zeros(len(nombres_cortos))
for (etapa, valores), color in zip(etapas_datos.items(), colores):
    ax1.bar(nombres_cortos, valores, bottom=bottom, label=etapa, color=color, alpha=0.85)
    bottom += np.array(valores)
ax1.set_title("Tiempo por Etapa y Tipo de Algoritmo")
ax1.set_ylabel("Tiempo (ms)")
ax1.legend(loc="upper left", fontsize=8)
ax1.tick_params(axis='x', rotation=15)

# Gráfico 2: Uso de memoria por algoritmo
ax2 = axes[0, 1]
nombres_mem = [n.split("_")[0] for n in complejidades_orden if n in memoria_resultados]
memorias = [memoria_resultados[c]["memoria_total_kb"] for c in complejidades_orden if c in memoria_resultados]
ax2.bar(nombres_mem, memorias, color="#55A868", alpha=0.85)
ax2.set_title("Uso de Memoria por Algoritmo")
ax2.set_ylabel("Memoria (KB)")
ax2.tick_params(axis='x', rotation=15)

# Gráfico 3: Distribución de tiempo (boxplot simulado con errores)
ax3 = axes[1, 0]
for i, nombre in enumerate(complejidades_orden):
    if nombre not in resultados_benchmark:
        continue
    datos = resultados_benchmark[nombre]
    media = datos["total_ms_media"]
    desv = datos["total_ms_desv"]
    ax3.errorbar(i, media, yerr=desv, fmt='o', capsize=5, 
                 color="#4C72B0", markersize=8)
ax3.set_xticks(range(len(complejidades_etiquetas)))
ax3.set_xticklabels(complejidades_etiquetas, rotation=15)
ax3.set_title("Tiempo Total con Desviación Estándar")
ax3.set_ylabel("Tiempo (ms)")

# Gráfico 4: Proporción de tiempo por etapa (pie)
ax4 = axes[1, 1]
tiempos_etapa_total = [
    sum(etapas_datos[e]) for e in etapas_datos
]
ax4.pie(tiempos_etapa_total, labels=list(etapas_datos.keys()),
        autopct='%1.1f%%', colors=colores, startangle=90)
ax4.set_title("Distribución de Tiempo por Etapa")

plt.tight_layout()
plt.savefig("performance_metrics.png", dpi=120, bbox_inches="tight")
plt.show()
print("Gráficos generados correctamente")

---

## 8. Resumen y Conclusiones

In [ ]:
print("RESUMEN DE MÉTRICAS DE PERFORMANCE")

# Tiempo total promedio
tiempos_todos = [datos["total_ms_media"] for datos in resultados_benchmark.values()]
print(f"\nTiempo promedio de análisis completo: {statistics.mean(tiempos_todos):.2f}ms")
print(f"Tiempo mínimo registrado:             {min(tiempos_todos):.2f}ms")
print(f"Tiempo máximo registrado:             {max(tiempos_todos):.2f}ms")

# Etapa más costosa
etapas_costo = {}
for etapa in ["parsing_ms", "analisis_ms", "patrones_ms", "estructuras_ms"]:
    costo_promedio = statistics.mean([
        datos[f"{etapa}_media"] for datos in resultados_benchmark.values()
    ])
    etapas_costo[etapa] = costo_promedio

etapa_mas_costosa = max(etapas_costo, key=etapas_costo.get)
print(f"\nEtapa más costosa: {etapa_mas_costosa} ({etapas_costo[etapa_mas_costosa]:.2f}ms promedio)")

# Memoria
memorias_todas = [datos["memoria_total_kb"] for datos in memoria_resultados.values()]
print(f"\nMemoria promedio por análisis: {statistics.mean(memorias_todas):.2f} KB")
print(f"Memoria máxima registrada:     {max(memorias_todas):.2f} KB")
print(f"\nThroughput del sistema: {throughput_actual:.1f} análisis/segundo")

---

## Proximos Pasos

- **accuracy_evaluation.ipynb**: Evaluar la precisión de los resultados de análisis
- **error_analysis.ipynb**: Analizar los casos de error y fallos del sistema